# MIP smoke test – forced rescheduling scenario

Deze notebook bouwt een **miniatuur-conflictscenario** met twee treinen op exact hetzelfde segment en tijdstip.

Doel:
- checken of het MIP-model effectief een herscheduling doet;
- inspecteren van `arrival`, `departure`, `ordering` en `delay`;
- snel debuggen zonder volledige gold timetable pipeline.

Het scenario:
- twee treinen willen tegelijk segment `A-B` betreden;
- beide segmenten hebben exclusieve bezetting;
- het MIP moet dus één trein laten wachten.


In [4]:
# Imports
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from domain.train import Train, TrainType, TrainSubtype
from domain.segment import Segment, SegmentType
from domain.schedule import Timetable, ScheduledTimes


from simulation.state import SystemState

from model.instance import build_instance
from model.solver import solve

import pandas as pd


## Fake netwerk

We bouwen:
- één conflictsegment: `A-B`
- twee treinen die exact hetzelfde pad gebruiken
- identieke geplande tijden

Dat MOET een conflict veroorzaken.


In [5]:
# Segmenten

segments = {
    "A-B": Segment(
        id="A-B",
        seg_type=SegmentType.BETWEEN_STATION,
        source="A",
        target="B",
    )
}

# Twee identieke treinen
trains = {
    1: Train(
        train_no=1,
        train_type=TrainType.PASSENGER,
        train_subtype=TrainSubtype.IC,
        path=("A-B",),
        halt_indicators={},
        dynamics={"A-B": "0-0"},
    ),

    2: Train(
        train_no=2,
        train_type=TrainType.FREIGHT,
        train_subtype=TrainSubtype.FREIGHT,
        path=("A-B",),
        halt_indicators={},
        dynamics={"A-B": "0-0"},
    ),
}


In [8]:
# Timetable

# Beide treinen willen exact tegelijk vertrekken:
# entry = 0
# exit  = 120

tt_data = {
    (1, "A-B"): ScheduledTimes(
        entry_seconds=0,
        exit_seconds=120,
        running_time=120,
        dwell_time=None,
    ),

    (2, "A-B"): ScheduledTimes(
        entry_seconds=0,
        exit_seconds=120,
        running_time=120,
        dwell_time=None,
    ),
}

timetable = Timetable(tt_data)
timetable


Timetable(2 treinen, 2 segmenten)

## SystemState

We simuleren dat beide treinen nog niet gestart zijn.


In [9]:
state = SystemState(
    trains=trains,
    timetable=timetable,
    start_time=0.0,
)

print(state)


SystemState(t=0s | actief=0, klaar=0, wachtend=2)


## Build instance

Hier zie je exact welke conflictset het MIP krijgt.


In [10]:
instance = build_instance(
    state=state,
    timetable=timetable,
    trains=trains,
    segments=segments,
    current_time=0.0,

    priority_strategy="static",

    weight_passenger=2,
    weight_freight=1,

    upgrade_weight=0,
    gamma=300,
)

print("T =", instance["T"])
print("S =", instance["S"])

print("\nConflict sets:")
for seg, pairs in instance["conflicts"].items():
    print(seg, pairs)


T = [1, 2]
S = {'A-B'}

Conflict sets:
A-B [(1, 2)]


## Solve

Normaal verwacht je:
- passenger trein eerst;
- freight trein vertraagd;
- `ordering[(1,2,"A-B")] = 1`


In [11]:
solution = solve(
    instance,
    priority_strategy="static",
    verbose=True,
)

solution


Set parameter Username
Set parameter LicenseID to value 2811974
Academic license - for non-commercial use only - expires 2027-04-22
[t=0] T=2 S=1 conf=1 vars=7 constr=6 bin=1


Solution(status=optimal, objective=120.00, runtime=0.01s)

In [14]:
print("STATUS:", solution.status)
print("OBJECTIVE:", solution.objective)

print("\nARRIVALS")
for k, v in solution.entry.items():
    print(k, v)

print("\nDEPARTURES")
for k, v in solution.entry.items():
    print(k, v)

print("\nDELAYS")
for k, v in solution.delay.items():
    print(k, v)




STATUS: optimal
OBJECTIVE: 120.0

ARRIVALS
(1, 'A-B') 0.0
(2, 'A-B') 120.0

DEPARTURES
(1, 'A-B') 0.0
(2, 'A-B') 120.0

DELAYS
(1, 'A-B') 0.0
(2, 'A-B') 120.0


## Mooie tabel

Hier kun je snel zien welke trein moest wachten.


In [17]:
rows = []

for (train_id, seg), arr in solution.entry.items():

    dep = solution.exit[(train_id, seg)]
    delay = solution.delay[(train_id, seg)]

    rows.append({
        "train": train_id,
        "segment": seg,
        "arrival": arr,
        "departure": dep,
        "delay": delay,
    })

df = pd.DataFrame(rows).sort_values("arrival")
df


,train,segment,arrival,departure,delay
0,1,A-B,0.0,120.0,0.0
1,2,A-B,120.0,240.0,120.0


## Verwachte interpretatie

Als alles correct werkt, krijg je ongeveer:

| trein | arrival | departure | delay |
|---|---:|---:|---:|
| 1 | 0 | 120 | 0 |
| 2 | 120 | 240 | 120 |

of omgekeerd als de ordering/warmstart anders zit.

Belangrijk:
- er mogen NOOIT overlappende bezettingstijden op hetzelfde segment zijn;
- exact één ordering variabele moet actief zijn.


## Uitbreidingen

Interessante volgende tests:
- voeg stationsegmenten toe;
- voeg dwell toe;
- test `dynamic` priority strategy;
- forceer `in_execution`;
- voeg headways toe;
- test deadlocks met meerdere segmenten.
